# Universal Olfactory NER Experiments

This notebook allows running Olfaction-inspired NER experiments across different datasets (CoNLL-2003, WikiANN, etc.) and languages using a unified framework.

**Architecture:**
- **Receptor Layer:** Specialized feature detectors inspired by olfactory receptors.
- **Glomerular Layer:** Aggregates receptor activations.
- **BiLSTM-CRF:** Standard backbone for NER.

In [ ]:
# @title 1. Configuration & Setup
# ==============================================================================

# ------------------------------------------------------------------------------
# USER INPUTS - CHANGE THESE
# ------------------------------------------------------------------------------
DATASET_KEY = 'conll_en'   # Options: 'conll_en', 'wikiann_mr'
MOUNT_DRIVE = True         # Set to True to save results to Google Drive

# Select experiments to run
EXPERIMENTS = [
    'baseline',            # Standard BiLSTM-CRF
    'activation_gelu',     # Standard Olfactory Model
    'olfactory_no_crf',    # Ablation: No CRF Layer
    'more_receptors'       # Larger Model
]
# ------------------------------------------------------------------------------

import os
import sys
import shutil
from google.colab import drive

# Mount Drive if requested
if MOUNT_DRIVE:
    drive.mount('/content/drive')
    BASE_SAVE_DIR = '/content/drive/My Drive/olfaction_inspired_ner'
else:
    BASE_SAVE_DIR = './results'

print(f"Results will be saved to: {BASE_SAVE_DIR}")

# Clone repository if not present (for Colab)
if not os.path.exists('olfaction-inspired-ner'):
    !git clone https://github.com/bhushan1729/olfaction-inspired-ner.git
    %cd olfaction-inspired-ner
else:
    %cd olfaction-inspired-ner
    !git pull

# Install dependencies
!pip install datasets pyyaml seqeval

# Add src to path
sys.path.append(os.getcwd())

In [ ]:
# @title 2. Run Experiments
# ==============================================================================
import yaml

# Verify config
CONFIG_PATH = 'config/universal_config.yaml'
with open(CONFIG_PATH, 'r') as f:
    config = yaml.safe_load(f)

if DATASET_KEY not in config['datasets']:
    raise ValueError(f"Dataset '{DATASET_KEY}' not found in unified config!")

dataset_info = config['datasets'][DATASET_KEY]
dataset_name = dataset_info['dataset']
language = dataset_info['language'] if dataset_info['language'] else 'default'

print(f"\n{'='*80}")
print(f"STARTING BATCH EXPERIMENTS")
print(f"Dataset: {dataset_name} ({language})")
print(f"Experiments: {EXPERIMENTS}")
print(f"{'='*80}\n")

for exp in EXPERIMENTS:
    print(f"\n>>> Running Experiment: {exp} <<<")
    
    !python src/train_universal.py \
        --config {CONFIG_PATH} \
        --dataset_key {DATASET_KEY} \
        --experiment {exp} \
        --save_dir "{BASE_SAVE_DIR}"
        
    print(f"✓ Completed {exp}")

In [ ]:
# @title 3. Generate Visualizations & Metrics
# ==============================================================================
import torch
import json
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from src.model.olfactory_ner import create_olfactory_ner
from src.data.unified_loader import get_dataset
from src.analysis.visualize import analyze_receptor_activations

# Setup paths
full_results_dir = os.path.join(BASE_SAVE_DIR, dataset_name, language)
viz_dir = os.path.join(full_results_dir, 'visualizations')
os.makedirs(viz_dir, exist_ok=True)

# Load Data (for analysis)
print("Loading test data for analysis...")
_, _, test_loader, vocab_info = get_dataset(
    dataset_name=dataset_name,
    language=dataset_info['language'],
    cache_dir='./data'
)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ------------------------------------------------------------------------------
# Generate Heatmaps (for Olfactory models)
# ------------------------------------------------------------------------------
print("\nGenerating Heatmaps...")
for exp in EXPERIMENTS:
    # Skip baselines or no-receptor models
    if 'baseline' in exp: continue
    
    model_path = os.path.join(full_results_dir, exp, 'best_model.pt')
    results_path = os.path.join(full_results_dir, exp, 'results.json')
    
    if not os.path.exists(model_path):
        print(f"Skipping {exp} (model not found)")
        continue
        
    # Load config and model
    with open(results_path, 'r') as f: 
        res = json.load(f)
    
    model_config = res['config']
    # Check if receptors enabled
    if not model_config.get('use_receptors', True):
        continue

    model = create_olfactory_ner(len(vocab_info['word2idx']), len(vocab_info['label2idx']), model_config)
    checkpoint = torch.load(model_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    model = model.to(device)
    
    save_path = os.path.join(viz_dir, exp)
    analyze_receptor_activations(model, test_loader, vocab_info, device, save_dir=save_path, experiment_name=exp)
    print(f"✓ Processed {exp}")

# ------------------------------------------------------------------------------
# Aggregate Metrics
# ------------------------------------------------------------------------------
metrics = []
for exp in EXPERIMENTS:
    res_path = os.path.join(full_results_dir, exp, 'results.json')
    if os.path.exists(res_path):
        with open(res_path, 'r') as f:
            data = json.load(f)
        
        test_res = data['test']
        metrics.append({
            'Experiment': exp,
            'F1': test_res['f1'],
            'Precision': test_res['precision'],
            'Recall': test_res['recall']
        })

df = pd.DataFrame(metrics)
print("\nResults Summary:")
print(df.sort_values(by='F1', ascending=False))

# Save summary
df.to_csv(os.path.join(viz_dir, 'summary_metrics.csv'), index=False)

In [ ]:
# @title 4. Compare Results (Plotting)
# ==============================================================================
import seaborn as sns

if not df.empty:
    plt.figure(figsize=(10, 6))
    sns.barplot(x='F1', y='Experiment', data=df.sort_values('F1', ascending=False), palette='viridis')
    plt.title(f'F1 Score Comparison - {dataset_name} ({language})')
    plt.xlim(0, 1.0)
    plt.grid(axis='x')
    plt.tight_layout()
    plt.savefig(os.path.join(viz_dir, 'f1_comparison.png'))
    plt.show()